In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q transformers datasets torch accelerate scipy
# ===================================================================

import os
import re
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import seaborn as sns
from transformers import BertTokenizer, BertModel
from IPython.display import display, HTML
from typing import List, Optional, Tuple
from matplotlib import rc
from scipy.stats import entropy as sp_entropy
from scipy.spatial.distance import jensenshannon
from scipy.stats import t

# Pillow check for saving GIFs
try:
    import PIL
    PIL_OK = True
except ImportError:
    PIL_OK = False


In [ ]:
# @title

class ReportReadyVisualizer:
    def __init__(self, model_name: str = "bert-base-uncased", save_dir: str = "attention_report", use_gpu: bool = True):
        print("--- Initializing ReportReadyVisualizer ---")
        self.model_name = model_name
        self.save_dir = save_dir
        os.makedirs(self.save_dir, exist_ok=True)
        print(f"Results will be saved to: {self.save_dir}")
        self.device = torch.device("cuda" if (use_gpu and torch.cuda.is_available()) else "cpu")
        print(f"Using device: {self.device}")
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.model = BertModel.from_pretrained(model_name, output_attentions=True).to(self.device)
        self.model.eval()
        self.L = self.model.config.num_hidden_layers
        self.H = self.model.config.num_attention_heads
        print(f"Model loaded: {self.L} layers, {self.H} heads per layer.")
        self.sentences: Optional[List[str]] = None
        self.attentions_list: Optional[List[np.ndarray]] = None  # list of [L,H,Q,K]
        self.tokens_list: Optional[List[List[str]]] = None
        self.base_unit = "bits"

    # --------------------- Basic Analysis ---------------------
    def analyze_sentences(self, sentences: List[str], max_len_tokens: Optional[int] = 128):
        print(f"\n--- Analyzing {len(sentences)} sentences ---")
        self.sentences = sentences
        inputs = self.tokenizer(sentences, padding=True, truncation=True, return_tensors="pt", max_length=max_len_tokens)
        self.all_inputs_cpu = {k: v.cpu() for k, v in inputs.items()}
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = self.model(**inputs)

        attentions_list_S = []
        self.tokens_list = []
        S = inputs['input_ids'].shape[0]
        for s_idx in range(S):
            eff_len = inputs['attention_mask'][s_idx].sum().item()

            tokens = self.tokenizer.convert_ids_to_tokens(inputs['input_ids'][s_idx])[:eff_len]
            self.tokens_list.append(tokens)

            sent_attentions_L = []
            for l_idx in range(self.L):
                attn_tensor = outputs.attentions[l_idx][s_idx, :, :eff_len, :eff_len].cpu().numpy()
                sent_attentions_L.append(attn_tensor)
            attentions_list_S.append(np.stack(sent_attentions_L, axis=0))  # [L,H,Q,K]
        self.attentions_list = attentions_list_S
        self.all_attentions_cpu = tuple(t.cpu() for t in outputs.attentions)
        print("Attention weights successfully extracted and stored as a list of arrays.")

    def _ensure_dir(self, sub_folder: str) -> str:
        d = os.path.join(self.save_dir, sub_folder)
        os.makedirs(d, exist_ok=True)
        return d

    # --------------------- Functions ---------------------
    @staticmethod
    def find_token_index(tokens: List[str], target: str, prefer_exact: bool = True) -> int:
        if prefer_exact:
            for i, t in enumerate(tokens):
                if t == target:
                    return i
        tgt = target.lower()
        for i, t in enumerate(tokens):
            if tgt in t.replace("##", "").lower():
                return i
        return max(0, min(len(tokens) - 1, 0))

    def calculate_layerwise_entropy_CI_heads(self, H_lh_mean, conf_level: float = 0.95):
        """
        Calculate Mean ± CI for each layer based on the "cross-head" approach.
        H_lh_mean: [L, H], the entropy of each layer × each head
        """
        mean = H_lh_mean.mean(axis=1)              # [L]
        std  = H_lh_mean.std(axis=1, ddof=1)       # [L]
        n = H_lh_mean.shape[1]                     # H
        sem = std / np.sqrt(n)
        df = n - 1
        tcrit = t.ppf(1 - (1 - conf_level) / 2, df)
        half = tcrit * sem
        lower, upper = mean - half, mean + half
        return mean, lower, upper, half

    def calculate_layerwise_entropy_CI_sentences(self, conf_level: float = 0.95):
        """
        Calculate Mean ± CI for each layer based on the "cross-sentence" approach:
        First, calculate the [mean of each layer × cross-head] for each sentence, then perform CI at the sentence level.
        """
        if self.attentions_list is None:
            raise ValueError("Please run analyze_sentences() first.")
        S = len(self.attentions_list)
        L = self.L

        per_sentence_layer_means = np.zeros((S, L), dtype=np.float64)
        for s in range(S):
            H_lh = self.calculate_entropy_matrix(s)   # [L, H]
            per_sentence_layer_means[s] = H_lh.mean(axis=1)

        mean = per_sentence_layer_means.mean(axis=0)        # [L]
        std  = per_sentence_layer_means.std(axis=0, ddof=1) # [L]
        n = per_sentence_layer_means.shape[0]               # 句子数
        sem = std / np.sqrt(n)
        df = n - 1
        tcrit = t.ppf(1 - (1 - conf_level) / 2, df)
        half = tcrit * sem
        lower, upper = mean - half, mean + half
        return mean, lower, upper, half
    # --------------------- Quantitative Analysis ---------------------
    def calculate_entropy_matrix(self, sentence_idx: int, base: int = 2):
        if self.attentions_list is None:
            raise ValueError("Please run analyze_sentences() first.")
        s_attn = self.attentions_list[sentence_idx]
        L, H, Q, K = s_attn.shape
        H_lh = np.zeros((L, H))
        for l in range(L):
            for h in range(H):
                head_entropies_q = [sp_entropy(s_attn[l, h, q, :], base=base) for q in range(Q)]
                H_lh[l, h] = np.mean(head_entropies_q)
        return H_lh

    def calculate_layerwise_entropy_stats(self, H_lh_mean):
        mean = H_lh_mean.mean(axis=1)
        std = H_lh_mean.std(axis=1, ddof=1)
        sem = std / np.sqrt(H_lh_mean.shape[1])
        return mean, sem

    def calculate_pairwise_jsd(self, layer_idx, head_idx, query_idx, base: int = 2, pad_mode: str = "right"):
        """
        For the same query_idx distribution, compare pairwise JSDs of different sentences.
        Zero-padding is applied to each distribution to the global maximum length to avoid broadcast errors.
        """
        if self.attentions_list is None:
            raise ValueError("Please run analyze_sentences() first.")

        vecs = []
        maxK = 0
        S = len(self.attentions_list)
        for s_idx in range(S):
            tokens = self.tokens_list[s_idx] # Get tokens for current sentence
            Q = len(tokens) # Use actual sequence length
            q_safe = min(query_idx, Q - 1) # Ensure query_idx is within bounds
            v = self.attentions_list[s_idx][layer_idx, head_idx, q_safe, :].astype(np.float64)
            v = np.clip(v, 0.0, None)
            s = v.sum()
            v = (np.ones_like(v) / len(v)) if s <= 0 else (v / s)
            vecs.append(v)
            maxK = max(maxK, v.shape[0])

        padded = []
        for v in vecs:
            if v.shape[0] < maxK:
                pad_len = maxK - v.shape[0]
                v = np.pad(v, (pad_len, 0), mode='constant', constant_values=0.0) if pad_mode == "left" \
                    else np.pad(v, (0, pad_len), mode='constant', constant_values=0.0)
            padded.append(v)

        jsd_matrix = np.zeros((S, S), dtype=np.float64)
        for i in range(S):
            for j in range(i, S):
                d = jensenshannon(padded[i], padded[j], base=base) ** 2
                jsd_matrix[i, j] = d
                jsd_matrix[j, i] = d
        return jsd_matrix


    def calculate_pairwise_jsd_by_token(self, layer_idx, head_idx, token_str: str, base: int = 2,
                                        prefer_exact: bool = True, pad_mode: str = "right"):
        """
        For the same token, the query index of that token is automatically located in each sentence, and then pairwise JSDs are calculated.
        Zero-padding is still performed to maintain consistent dimensions.
        """
        if self.attentions_list is None:
            raise ValueError("Please run analyze_sentences() first.")
        vecs, maxK = [], 0
        S = len(self.attentions_list)
        for s_idx in range(S):
            tokens = self.tokens_list[s_idx]
            q = self.find_token_index(tokens, token_str, prefer_exact=prefer_exact)
            v = self.attentions_list[s_idx][layer_idx, head_idx, q, :].astype(np.float64)
            v = np.clip(v, 0.0, None)
            s = v.sum()
            v = (np.ones_like(v) / len(v)) if s <= 0 else (v / s)
            vecs.append(v)
            maxK = max(maxK, v.shape[0])

        padded = []
        for v in vecs:
            if v.shape[0] < maxK:
                pad_len = maxK - v.shape[0]
                v = np.pad(v, (pad_len, 0), mode='constant', constant_values=0.0) if pad_mode == "left" \
                    else np.pad(v, (0, pad_len), mode='constant', constant_values=0.0)
            padded.append(v)

        jsd_matrix = np.zeros((S, S), dtype=np.float64)
        for i in range(S):
            for j in range(i, S):
                d = jensenshannon(padded[i], padded[j], base=base) ** 2
                jsd_matrix[i, j] = d
                jsd_matrix[j, i] = d
        return jsd_matrix

    def pick_heads_by_entropy(self, H_lh_layer):
        order = np.argsort(H_lh_layer)
        low = order[0]
        high = order[-1]
        mid = order[np.argmin(np.abs(H_lh_layer - np.median(H_lh_layer)))]
        return low, mid, high

    # --------------------- Core Charts ---------------------
    def plot_figure_1_global_entropy_heatmap(self, force: bool = False, show=True):
        print("\n--- Generating Figure 1: Global Attention Entropy Heatmap ---")
        filepath = os.path.join(self.save_dir, "Figure_1_Global_Entropy_Heatmap.png")
        if os.path.exists(filepath) and not force:
            print(f"Skipping, already exists: {filepath}"); return
        if self.attentions_list is None:
            raise ValueError("Please run analyze_sentences() first.")

        H_lh_all_sentences = [self.calculate_entropy_matrix(s_idx) for s_idx in range(len(self.sentences))]
        H_lh_mean = np.mean(H_lh_all_sentences, axis=0)

        plt.figure(figsize=(10, 8))
        ax = sns.heatmap(
            H_lh_mean, annot=True, fmt=".2f", cmap="viridis",
            xticklabels=[f'H{i+1}' for i in range(self.H)],
            yticklabels=[f'L{i+1}' for i in range(self.L)],
            cbar_kws={'label': 'Attention Entropy (bits)'}
        )
        plt.title('Figure 1: Global Average Attention Entropy (Layer × Head)', fontsize=16)
        plt.xlabel('Heads')
        plt.ylabel('Layers')
        plt.tight_layout()
        plt.savefig(filepath)
        if show: plt.show()
        else: plt.close()
        print(f"Saved: {filepath}")
        return H_lh_mean

    def plot_figure_2_layerwise_entropy_stats(self, H_lh_mean, force: bool = False, show=True):
        print("--- Generating Figure 2: Layer-wise Entropy Statistics ---")
        filepath = os.path.join(self.save_dir, "Figure_2_Layerwise_Entropy_Stats.png")
        if os.path.exists(filepath) and not force:
            print(f"Skipping, already exists: {filepath}"); return
        mean, sem = self.calculate_layerwise_entropy_stats(H_lh_mean)

        x = np.arange(len(mean)) + 1
        plt.figure(figsize=(8, 5))
        plt.plot(x, mean, linewidth=2, marker='o')
        plt.fill_between(x, mean - sem, mean + sem, alpha=0.2)
        plt.xlabel('Layer')
        plt.ylabel('Attention Entropy (bits)')
        plt.title('Figure 2: Per-layer Attention Entropy (Mean ± SEM)', fontsize=14)
        plt.xticks(x)
        plt.grid(True, linestyle='--')
        plt.tight_layout()
        plt.savefig(filepath)
        if show: plt.show()
        else: plt.close()
        print(f"Saved: {filepath}")

    def plot_figure_2_layerwise_entropy_stats_ci(self, H_lh_mean, force: bool = False, show=True,
                                          conf_level: float = 0.95, ci_basis: str = "heads"):
        """
       Mean ± CI, supports two methods:
       - ci_basis="heads": Perform CI on different heads within the same layer (default)
       - ci_basis="sentences": Perform CI on the mean of different sentence heads (more like generalization stability)
        """
        title_basis = "across Heads" if ci_basis == "heads" else "across Sentences"
        filepath = os.path.join(self.save_dir, f"Figure_2_Layerwise_Entropy_{int(conf_level*100)}CI_{ci_basis}.png")
        if os.path.exists(filepath) and not force:
            print(f"Skipping, already exists: {filepath}"); return

        if ci_basis == "heads":
            mean, lower, upper, half = self.calculate_layerwise_entropy_CI_heads(H_lh_mean, conf_level=conf_level)
        elif ci_basis == "sentences":
            mean, lower, upper, half = self.calculate_layerwise_entropy_CI_sentences(conf_level=conf_level)
        else:
            raise ValueError("ci_basis must be 'heads' or 'sentences'")

        x = np.arange(len(mean)) + 1
        plt.figure(figsize=(8, 5))
        plt.plot(x, mean, linewidth=2, marker='o', label='Mean')
        plt.fill_between(x, lower, upper, alpha=0.2, label=f'{int(conf_level*100)}% CI')
        plt.xlabel('Layer'); plt.ylabel('Attention Entropy (bits)')
        plt.title(f'Figure 2: Per-layer Attention Entropy (Mean ± {int(conf_level*100)}% CI, {title_basis})', fontsize=14)
        plt.xticks(x); plt.grid(True, linestyle='--'); plt.legend()
        plt.tight_layout(); plt.savefig(filepath)
        if show: plt.show()
        else: plt.close()
        print(f"Saved: {filepath}")


    def plot_figure_case_study(self, sent_idx, query_idx, layer_idx, head_idx, phenomenon_name, force: bool = False, show=True):
        print(f"--- Generating Figure for Case Study: {phenomenon_name} ---")
        filepath = os.path.join(self.save_dir, f"Figure_Case_{phenomenon_name.replace(' ', '_').replace('->', 'to')}.png")
        if os.path.exists(filepath) and not force:
            print(f"Skipping, already exists: {filepath}"); return

        tokens = self.tokens_list[sent_idx]
        Q = len(tokens)
        q_safe = min(query_idx, Q - 1)
        attn_vec = self.attentions_list[sent_idx][layer_idx, head_idx, q_safe, :]

        plt.figure(figsize=(8, 4))
        plt.plot(np.arange(Q), attn_vec, marker='o', linestyle='-')
        plt.xticks(np.arange(Q), tokens, rotation=90)
        plt.ylabel("Attention Weight")
        plt.title(f'Case Study: {phenomenon_name}\nAttention from "{tokens[q_safe]}" (L{layer_idx+1}-H{head_idx+1})')
        plt.grid(True, linestyle='--')
        plt.tight_layout()
        plt.savefig(filepath)
        if show: plt.show()
        else: plt.close()
        print(f"Saved: {filepath}")


    def plot_figure_jsd_comparison(self, jsd_matrix, sentence_groups, title_suffix, force: bool = False, show=True):
        print(f"--- Generating JSD Comparison Figure for: {title_suffix} ---")
        filepath = os.path.join(self.save_dir, f"Figure_JSD_Comparison_{title_suffix.replace(' ', '_')}.png")
        if os.path.exists(filepath) and not force:
            print(f"Skipping, already exists: {filepath}"); return

        S = len(sentence_groups)
        intra, inter = [], []
        for i in range(S):
            for j in range(i + 1, S):
                if sentence_groups[i] == "ignore" or sentence_groups[j] == "ignore":
                    continue
                (intra if sentence_groups[i] == sentence_groups[j] else inter).append(jsd_matrix[i, j])

        if len(intra) == 0 and len(inter) == 0:
            print("[WARN] No valid pairs for JSD comparison; skipping figure.")
            return

        plt.figure(figsize=(6, 5))
        data, labels = [], []
        if len(intra): data.append(np.array(intra)); labels.append('Within-Group')
        if len(inter): data.append(np.array(inter)); labels.append('Across-Groups')

        plt.violinplot(data, showmeans=True)
        plt.xticks(np.arange(1, len(data) + 1), labels)
        plt.ylabel('Jensen-Shannon Divergence (JSD, bits)')
        plt.title(f'JSD Comparison: {title_suffix}')
        plt.grid(True, axis='y')
        plt.tight_layout()
        plt.savefig(filepath)
        if show: plt.show()
        else: plt.close()
        print(f"Saved: {filepath}")

    # --------------------- Animation ---------------------
    def animate_in_main_dir(self, sent_idx: int, anim_type: str, phenomenon_name: str, force: bool = False, **kwargs):
        """
        Generates an output file with the phenomenon name in the main report directory.
        Supporting:
          - anim_type='query_to_others' | 'key_from_queries'
          - mode='LH-mean' (static PNG) | 'per-layer' (GIF)
        """
        print(f"--- Generating report figure: {phenomenon_name} ---")

        mode = kwargs.get('mode', 'per-layer')
        if mode not in ('LH-mean', 'per-layer'):
            print(f"Error: Unknown mode '{mode}'. Use 'LH-mean' or 'per-layer'."); return

        extension = "png" if mode == "LH-mean" else "gif"
        final_filename = os.path.join(self.save_dir, f"Figure_{phenomenon_name.replace(' ', '_').replace('->','to')}.{extension}")

        if os.path.exists(final_filename) and not force:
            print(f"Skipping, already exists: {final_filename}")
            return

        try:
            if anim_type == 'query_to_others':
                self.animate_query_to_others(sent_idx=sent_idx, query_idx=kwargs.get('query_idx'), mode=mode, force=force)
            elif anim_type == 'key_from_queries':
                self.animate_key_from_queries(sent_idx=sent_idx, key_idx=kwargs.get('key_idx'), mode=mode, force=force)
            else:
                print(f"Error: Unknown anim_type '{anim_type}'."); return
        except Exception as e:
            print(f"An error occurred during underlying generation: {e}")
            return

        original_sub_dir = self._ensure_dir(f"appendix/sentence_{sent_idx+1}/animations")
        if anim_type == 'query_to_others':
            q_safe = min(kwargs.get('query_idx', 0), len(self.tokens_list[sent_idx]) - 1)
            fname_mode = "LHmean" if mode == "LH-mean" else "perLayer"
            original_filename = os.path.join(original_sub_dir, f"S{sent_idx+1}_Q{q_safe}_query_to_keys_{fname_mode}.{extension}")
        else:
            k_safe = min(kwargs.get('key_idx', 0), len(self.tokens_list[sent_idx]) - 1)
            fname_mode = "LHmean" if mode == "LH-mean" else "perLayer"
            original_filename = os.path.join(original_sub_dir, f"S{sent_idx+1}_K{k_safe}_queries_to_key_{fname_mode}.{extension}")

        if os.path.exists(original_filename):
            import shutil
            if force and os.path.exists(final_filename):
                try: os.remove(final_filename)
                except Exception: pass
            shutil.move(original_filename, final_filename)
            print(f"Saved and moved to: {final_filename}")
        else:
            print(f"File may have been displayed inline or was skipped. Not found: '{original_filename}'")

    # --------------------- Appendix/Exploratory Drawing ---------------------
    def _apply_gif_layout(self, fig, top=0.95): # Increased top value
        try:
            fig.tight_layout(rect=[0, 0, 1, top])
        except Exception:
            plt.subplots_adjust(top=top)

    def plot_global_mass_to_token(self, token_str: str = "[CLS]", force: bool = False, show: bool = False):
        sub_dir = self._ensure_dir("appendix/global_stats")
        filepath = os.path.join(sub_dir, f"global_mass_to_{token_str.replace('[', '').replace(']', '')}.png")
        if os.path.exists(filepath) and not force:
            print(f"Skipping, already exists: {filepath}"); return

        token_id = self.tokenizer.convert_tokens_to_ids(token_str)
        M = np.zeros((self.L, self.H), dtype=np.float32); B = self.all_attentions_cpu[0].shape[0]
        for l in range(self.L):
            A_layer = self.all_attentions_cpu[l]; layer_mass = []
            for b in range(B):
                ids = self.all_inputs_cpu["input_ids"][b].tolist()
                cols = [i for i, t in enumerate(ids) if t == token_id]
                if not cols: continue
                mass = A_layer[b, :, :, cols].sum(dim=-1).mean(dim=-1); layer_mass.append(mass)
            if layer_mass: M[l] = torch.stack(layer_mass).mean(0).numpy()
        plt.figure(figsize=(10, 6))
        sns.heatmap(M, cmap="magma",
                    xticklabels=[f"H{i+1}" for i in range(self.H)],
                    yticklabels=[f"L{i+1}" for i in range(self.L)],
                    cbar_kws={'label': f'Avg mass -> {token_str}'})
        plt.title(f'Global Layer-Head Attention to {token_str}')
        plt.xlabel("Heads"); plt.ylabel("Layers")
        plt.tight_layout(); plt.savefig(filepath)
        if show: plt.show()
        else: plt.close(); print(f"Saved: {filepath}")

    def plot_global_specialization(self, force: bool = False, show: bool = False):
        sub_dir = self._ensure_dir("appendix/global_stats")
        filepath = os.path.join(sub_dir, "global_specialization_matrix.png")
        if os.path.exists(filepath) and not force:
            print(f"Skipping, already exists: {filepath}"); return
        M = np.zeros((self.L, self.H), dtype=np.float32)
        for l in range(self.L):
            A = self.all_attentions_cpu[l]; max_per_query = A.max(dim=-1).values
            mean_max_prob = max_per_query.mean(dim=(0, 2)); M[l] = mean_max_prob.numpy()
        plt.figure(figsize=(10, 6))
        sns.heatmap(M, cmap="coolwarm",
                    xticklabels=[f"H{i+1}" for i in range(self.H)],
                    yticklabels=[f"L{i+1}" for i in range(self.L)],
                    cbar_kws={'label': 'Specialization (mean max prob)'})
        plt.title("Global Layer-Head Specialization")
        plt.xlabel("Heads"); plt.ylabel("Layers")
        plt.tight_layout(); plt.savefig(filepath)
        if show: plt.show()
        else: plt.close(); print(f"Saved: {filepath}")

    def plot_all_head_similarities(self, force: bool = False):
        print("\n--- Generating All Head Similarity plots (for Appendix) ---")
        if self.sentences is None:
            print("    [ERROR] Please run analyze_sentences() first."); return
        for sent_idx in range(len(self.sentences)):
            print(f"  Processing Sentence {sent_idx+1}...")
            for layer_idx in range(self.L):
                self._plot_head_similarity_in_layer(sent_idx, layer_idx, force=force)

    def _plot_head_similarity_in_layer(self, sent_idx: int, layer_idx: int, force: bool = False, show: bool = False):
        sub_dir = self._ensure_dir(f"appendix/sentence_{sent_idx+1}/similarity")
        filepath = os.path.join(sub_dir, f"S{sent_idx+1}_L{layer_idx+1}_head_similarity.png")
        if os.path.exists(filepath) and not force:
            print(f"Skipping, already exists: {filepath}"); return

        A = self.attentions_list[sent_idx][layer_idx]
        H, Q, K = A.shape
        X = A.reshape(H, Q * K)
        X_norm = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
        S = X_norm @ X_norm.T

        plt.figure(figsize=(7, 6))
        sns.heatmap(S, vmin=0, vmax=1, cmap="viridis",
                    xticklabels=[f"H{i+1}" for i in range(H)],
                    yticklabels=[f"H{i+1}" for i in range(H)],
                    cbar_kws={'label': 'Cosine Similarity'})
        plt.title(f"Head Similarity (Layer {layer_idx+1}) for Sentence {sent_idx+1}")
        plt.tight_layout(); plt.savefig(filepath)
        if show: plt.show()
        else: plt.close()
        print(f"Saved: {filepath}")

    def plot_all_layer_grids(self, force: bool = False):
        print("\n--- Generating All Layer Grids (for Appendix) ---")
        if self.sentences is None:
            print("    [ERROR] Please run analyze_sentences() first."); return
        for sent_idx in range(len(self.sentences)):
            print(f"  Processing Sentence {sent_idx+1}...")
            for layer_idx in range(self.L):
                self._plot_layer_all_heads_grid(sent_idx, layer_idx, force=force)

    def _plot_layer_all_heads_grid(self, sent_idx: int, layer_idx: int, force: bool = False, show: bool = False):
        sub_dir = self._ensure_dir(f"appendix/sentence_{sent_idx+1}/grids")
        filepath = os.path.join(sub_dir, f"S{sent_idx+1}_L{layer_idx+1}_all_heads_grid.png")
        if os.path.exists(filepath) and not force:
            print(f"Skipping, already exists: {filepath}"); return

        tokens = self.tokens_list[sent_idx]
        Q = len(tokens)
        A = self.attentions_list[sent_idx][layer_idx]
        vmin, vmax = float(A.min()), float(A.max())
        cols, rows = 4, int(np.ceil(self.H / 4))
        fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 4), constrained_layout=False)
        axes = axes.flatten()
        for h_idx in range(self.H):
            ax = axes[h_idx]
            im = ax.imshow(A[h_idx], vmin=vmin, vmax=vmax, cmap='viridis', aspect='auto')
            ax.set_title(f'Head {h_idx+1}', fontsize=10)
            ax.set_xticks(np.arange(Q)); ax.set_xticklabels(tokens, rotation=90, fontsize=6)
            ax.set_yticks(np.arange(Q)); ax.set_yticklabels(tokens, fontsize=6)
        for ax in axes[self.H:]:
            ax.set_visible(False)
        fig.suptitle(f'Layer {layer_idx+1} Attention Heads\n"{self.sentences[sent_idx]}"', fontsize=16)
        fig.colorbar(im, ax=axes.tolist(), shrink=0.6, label="Attention Weight")
        fig.tight_layout(rect=[0, 0, 1, 0.92])
        plt.savefig(filepath)
        if show: plt.show()
        else: plt.close(fig)
        print(f"Saved: {filepath}")

    # --------------------- Various Animation ---------------------
    def animate_heads_in_layer(self, sent_idx: int, layer_idx: int, force: bool = False, interval: int = 600, fps: int = 2):
        sub_dir = self._ensure_dir(f"appendix/sentence_{sent_idx+1}/animations")
        filename = os.path.join(sub_dir, f"S{sent_idx+1}_L{layer_idx+1}_headsHeatmap.gif")
        if os.path.exists(filename) and not force:
            print(f"Skipping, already exists: {filename}"); return

        tokens = self.tokens_list[sent_idx]
        Q = len(tokens)
        A = self.attentions_list[sent_idx][layer_idx]
        vmin, vmax = float(A.min()), float(A.max())

        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(A[0], vmin=vmin, vmax=vmax, cmap='viridis', aspect='auto')
        plt.colorbar(im, ax=ax).set_label("Attention Weight")
        ax.set_xticks(np.arange(Q)); ax.set_xticklabels(tokens, rotation=90)
        ax.set_yticks(np.arange(Q)); ax.set_yticklabels(tokens)
        title = ax.set_title(f'Layer {layer_idx+1} Head 1\n"{self.sentences[sent_idx]}"')
        self._apply_gif_layout(fig, top=0.90)

        def update(h):
            im.set_data(A[h])
            title.set_text(f'Layer {layer_idx+1} Head {h+1}\n"{self.sentences[sent_idx]}"')
            return [im, title]

        ani = animation.FuncAnimation(fig, update, frames=self.H, interval=interval, blit=False)
        if PIL_OK: ani.save(filename, writer='pillow', fps=fps); print(f"Saved: {filename}")
        else: display(HTML(ani.to_jshtml()))
        plt.close(fig)

    def animate_layers_for_head(self, sent_idx: int, head_idx: int, force: bool = False, interval: int = 600, fps: int = 2):
        sub_dir = self._ensure_dir(f"appendix/sentence_{sent_idx+1}/animations")
        filename = os.path.join(sub_dir, f"S{sent_idx+1}_H{head_idx+1}_layersHeatmap.gif")
        if os.path.exists(filename) and not force:
            print(f"Skipping, already exists: {filename}"); return

        tokens = self.tokens_list[sent_idx]
        Q = len(tokens)
        frames = self.attentions_list[sent_idx][:, head_idx, :, :]
        vmin, vmax = float(frames.min()), float(frames.max())

        fig, ax = plt.subplots(figsize=(8, 6)); im = ax.imshow(frames[0], vmin=vmin, vmax=vmax, cmap='viridis', aspect='auto')
        plt.colorbar(im, ax=ax).set_label("Attention Weight"); ax.set_xticks(np.arange(Q)); ax.set_xticklabels(tokens, rotation=90)
        ax.set_yticks(np.arange(Q)); ax.set_yticklabels(tokens); title = ax.set_title(f'Layer 1 Head {head_idx+1}\n"{self.sentences[sent_idx]}"'); self._apply_gif_layout(fig, top=0.90)
        def update(l): im.set_data(frames[l]); title.set_text(f'Layer {l+1} Head {head_idx+1}\n"{self.sentences[sent_idx]}"'); return [im, title]
        ani = animation.FuncAnimation(fig, update, frames=self.L, interval=interval, blit=False)
        if PIL_OK: ani.save(filename, writer='pillow', fps=fps); print(f"Saved: {filename}")
        else: display(HTML(ani.to_jshtml()));
        plt.close(fig)

    def animate_query_to_others(self, sent_idx: int, query_idx: int, mode: str = "per-layer", force: bool = False):
        sub_dir = self._ensure_dir(f"appendix/sentence_{sent_idx+1}/animations")
        tokens = self.tokens_list[sent_idx]
        Q = len(tokens)
        q_safe = min(query_idx, Q-1)

        if mode == "LH-mean":
            filepath = os.path.join(sub_dir, f"S{sent_idx+1}_Q{q_safe}_query_to_keys_LHmean.png")
            if os.path.exists(filepath) and not force:
                print(f"Skipping, already exists: {filepath}"); return
            vec = self.attentions_list[sent_idx][:, :, q_safe, :].mean(axis=(0,1))
            plt.figure(figsize=(10, 3.8))
            plt.plot(np.arange(Q), vec, marker='o')
            plt.ylim(0, max(1e-6, vec.max()) * 1.1)
            plt.xticks(np.arange(Q), tokens, rotation=90)
            plt.title(f'Query "{tokens[q_safe]}" → Keys (LH-mean)')
            plt.xlabel("Key"); plt.ylabel("Avg Attention")
            plt.grid(axis='y', linestyle='--', alpha=0.7)
            plt.tight_layout()
            plt.savefig(filepath)
            plt.close()
            print(f"Saved: {filepath}")
            return

        if mode == "per-layer":
            filename = os.path.join(sub_dir, f"S{sent_idx+1}_Q{q_safe}_query_to_keys_perLayer.gif")
            if os.path.exists(filename) and not force:
                print(f"Skipping, already exists: {filename}"); return
            frames, ymax = [], 1e-8
            for l in range(self.L):
                y = self.attentions_list[sent_idx][l, :, q_safe, :].mean(axis=0)
                frames.append(y); ymax = max(ymax, float(y.max()))
            fig, ax = plt.subplots(figsize=(10, 3.8))
            (line,) = ax.plot([], [], marker='o')
            ax.set_xlim(-0.5, Q - 0.5); ax.set_ylim(0, ymax * 1.1)
            ax.set_xticks(np.arange(Q)); ax.set_xticklabels(tokens, rotation=90)
            ax.set_xlabel("Key"); ax.set_ylabel("Avg Attention per Layer")
            ax.grid(axis='y', linestyle='--', alpha=0.7)
            title = ax.set_title(f'Layer 1 — Query "{tokens[q_safe]}" → Keys')
            self._apply_gif_layout(fig, top=0.90)

            def update(l):
                line.set_data(np.arange(Q), frames[l])
                title.set_text(f'Layer {l+1} — Query "{tokens[q_safe]}" → Keys')
                return [line, title]

            ani = animation.FuncAnimation(fig, update, frames=self.L, interval=600, blit=False)
            if PIL_OK: ani.save(filename, writer='pillow', fps=2); print(f"Saved: {filename}")
            else: display(HTML(ani.to_jshtml()))
            plt.close(fig); return

    def animate_key_from_queries(self, sent_idx: int, key_idx: int, mode: str = "per-layer", force: bool = False):
        sub_dir = self._ensure_dir(f"appendix/sentence_{sent_idx+1}/animations")
        tokens = self.tokens_list[sent_idx]
        Q = len(tokens)
        k_safe = min(key_idx, Q-1)

        if mode == "LH-mean":
            filepath = os.path.join(sub_dir, f"S{sent_idx+1}_K{k_safe}_queries_to_key_LHmean.png")
            if os.path.exists(filepath) and not force:
                print(f"Skipping, already exists: {filepath}"); return
            vec = self.attentions_list[sent_idx][:, :, :, k_safe].mean(axis=(0,1))
            plt.figure(figsize=(10, 3.8))
            plt.plot(np.arange(Q), vec, marker='o')
            plt.ylim(0, max(1e-6, vec.max()) * 1.1)
            plt.xticks(np.arange(Q), tokens, rotation=90)
            plt.title(f'Queries → Key "{tokens[k_safe]}" (LH-mean)')
            plt.xlabel("Query"); plt.ylabel("Avg Attention")
            plt.grid(axis='y', linestyle='--', alpha=0.7)
            plt.tight_layout()
            plt.savefig(filepath)
            plt.close()
            print(f"Saved: {filepath}")
            return

        if mode == "per-layer":
            filename = os.path.join(sub_dir, f"S{sent_idx+1}_K{k_safe}_queries_to_key_perLayer.gif")
            if os.path.exists(filename) and not force:
                print(f"Skipping, already exists: {filename}"); return
            frames, ymax = [], 1e-8
            for l in range(self.L):
                y = self.attentions_list[sent_idx][l, :, :, k_safe].mean(axis=0)
                frames.append(y); ymax = max(ymax, float(y.max()))
            fig, ax = plt.subplots(figsize=(10, 3.8))
            (line,) = ax.plot([], [], marker='o')
            ax.set_xlim(-0.5, Q - 0.5); ax.set_ylim(0, ymax * 1.1)
            ax.set_xticks(np.arange(Q)); ax.set_xticklabels(tokens, rotation=90)
            ax.set_xlabel("Query"); ax.set_ylabel("Avg Attention per Layer")
            ax.grid(axis='y', linestyle='--', alpha=0.7)
            title = ax.set_title(f'Layer 1 — Queries → Key "{tokens[k_safe]}"')
            self._apply_gif_layout(fig, top=0.90)

            def update(l):
                line.set_data(np.arange(Q), frames[l])
                title.set_text(f'Layer {l+1} — Queries → Key "{tokens[k_safe]}"')
                return [line, title]

            ani = animation.FuncAnimation(fig, update, frames=self.L, interval=600, blit=False)
            if PIL_OK: ani.save(filename, writer='pillow', fps=2); print(f"Saved: {filename}")
            else: display(HTML(ani.to_jshtml()))
            plt.close(fig); return

    def animate_query_entropy_heatmap(self, sent_idx: int, force: bool = False, interval: int = 700, fps: int = 2):
        sub_dir = self._ensure_dir(f"appendix/sentence_{sent_idx+1}/animations")
        filename = os.path.join(sub_dir, f"S{sent_idx+1}_LH_entropy_perQuery.gif")
        if os.path.exists(filename) and not force:
            print(f"Skipping, already exists: {filename}"); return

        tokens = self.tokens_list[sent_idx]
        Q = len(tokens)
        A = self.attentions_list[sent_idx]
        ent_frames = np.zeros((Q, self.L, self.H))
        for q_idx in range(Q):
            for l_idx in range(self.L):
                for h_idx in range(self.H):
                    ent_frames[q_idx, l_idx, h_idx] = sp_entropy(A[l_idx, h_idx, q_idx, :], base=2)

        vmin, vmax = float(ent_frames.min()), float(ent_frames.max())
        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(ent_frames[0], vmin=vmin, vmax=vmax, cmap='magma', aspect='auto')
        plt.colorbar(im, ax=ax).set_label(f"Entropy ({self.base_unit})")
        ax.set_xticks(np.arange(self.H)); ax.set_xticklabels([f"H{i+1}" for i in range(self.H)])
        ax.set_yticks(np.arange(self.L)); ax.set_yticklabels([f"L{i+1}" for i in range(self.L)])
        ax.set_xlabel("Heads"); ax.set_ylabel("Layers")
        title = ax.set_title(f'Query[0]: {tokens[0]}\n"{self.sentences[sent_idx]}"')
        self._apply_gif_layout(fig, top=0.90)

        def update(q):
            im.set_data(ent_frames[q])
            title.set_text(f'Query[{q}]: {tokens[q]}\n"{self.sentences[sent_idx]}"')
            return [im, title]

        ani = animation.FuncAnimation(fig, update, frames=Q, interval=interval, blit=False)
        if PIL_OK: ani.save(filename, writer='pillow', fps=fps); print(f"Saved: {filename}")
        else: display(HTML(ani.to_jshtml()))
        plt.close(fig)

    def animate_key_mass_heatmap(self, sent_idx: int, force: bool = False, interval: int = 700, fps: int = 2):
        sub_dir = self._ensure_dir(f"appendix/sentence_{sent_idx+1}/animations")
        filename = os.path.join(sub_dir, f"S{sent_idx+1}_LH_mass_perKey.gif")
        if os.path.exists(filename) and not force:
            print(f"Skipping, already exists: {filename}"); return

        tokens = self.tokens_list[sent_idx]
        Q = len(tokens)
        A = self.attentions_list[sent_idx]
        mass_frames = np.zeros((Q, self.L, self.H))
        for k_idx in range(Q):
            mass_frames[k_idx, :, :] = A[:, :, :, k_idx].mean(axis=2)

        vmin, vmax = float(mass_frames.min()), float(mass_frames.max())
        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(mass_frames[0], vmin=vmin, vmax=vmax, cmap='plasma', aspect='auto')
        plt.colorbar(im, ax=ax).set_label("Avg Attention Received by this Token")
        ax.set_xticks(np.arange(self.H)); ax.set_xticklabels([f"H{i+1}" for i in range(self.H)])
        ax.set_yticks(np.arange(self.L)); ax.set_yticklabels([f"L{i+1}" for i in range(self.L)])
        ax.set_xlabel("Heads"); ax.set_ylabel("Layers")
        title = ax.set_title(f'Key[0]: "{tokens[0]}" - Attention Received\n"{self.sentences[sent_idx]}"')
        self._apply_gif_layout(fig, top=0.90)

        def update(k):
            im.set_data(mass_frames[k])
            title.set_text(f'Key[{k}]: "{tokens[k]}" - Attention Received\n"{self.sentences[sent_idx]}"')
            return [im, title]

        ani = animation.FuncAnimation(fig, update, frames=Q, interval=interval, blit=False)
        if PIL_OK: ani.save(filename, writer='pillow', fps=fps); print(f"Saved: {filename}")
        else: display(HTML(ani.to_jshtml()))
        plt.close(fig)

    def animate_head_entropy_curves(self, sent_idx: int, layer_idx: int, force: bool = False, interval: int = 600, fps: int = 2):
        sub_dir = self._ensure_dir(f"appendix/sentence_{sent_idx+1}/animations")
        filename = os.path.join(sub_dir, f"S{sent_idx+1}_L{layer_idx+1}_head_entropy_curves.gif")
        if os.path.exists(filename) and not force:
            print(f"Skipping, already exists: {filename}"); return

        tokens = self.tokens_list[sent_idx]
        Q = len(tokens)
        A = self.attentions_list[sent_idx][layer_idx]
        ent = np.array([[sp_entropy(A[h, q, :], base=2) for q in range(Q)] for h in range(self.H)])

        ymin, ymax = float(ent.min()), float(ent.max())
        fig, ax = plt.subplots(figsize=(10, 4))
        (line,) = ax.plot([], [], marker='o')
        ax.set_xlim(-0.5, Q - 0.5); ax.set_ylim(ymin - 0.1, ymax + 0.1)
        ax.set_xticks(np.arange(Q)); ax.set_xticklabels(tokens, rotation=90)
        ax.set_xlabel("Token (Query)"); ax.set_ylabel(f"Entropy ({self.base_unit})")
        title = ax.set_title(f'Layer {layer_idx+1} Head 1 (Avg Entropy)')
        self._apply_gif_layout(fig, top=0.90)

        def update(h):
            y_data = ent[h]
            line.set_data(np.arange(Q), y_data)
            title.set_text(f'Layer {layer_idx+1} Head {h+1} (Avg Entropy: {y_data.mean():.2f})\n"{self.sentences[sent_idx]}"')
            return [line, title]

        ani = animation.FuncAnimation(fig, update, frames=self.H, interval=interval, blit=False)
        if PIL_OK: ani.save(filename, writer='pillow', fps=fps); print(f"Saved: {filename}")
        else: display(HTML(ani.to_jshtml()))
        plt.close(fig)

    def animate_per_query_all_heads_overlay(self, sent_idx: int, layer_idx: int, force: bool = False, interval: int = 800, fps: int = 2):
        sub_dir = self._ensure_dir(f"appendix/sentence_{sent_idx+1}/animations")
        filename = os.path.join(sub_dir, f"S{sent_idx+1}_L{layer_idx+1}_perQuery_allHeads.gif")
        if os.path.exists(filename) and not force:
            print(f"Skipping, already exists: {filename}"); return

        tokens = self.tokens_list[sent_idx]
        Q = len(tokens)
        A = self.attentions_list[sent_idx][layer_idx]
        frames = A.transpose(1, 0, 2)
        ymax = float(frames.max())

        fig, ax = plt.subplots(figsize=(10, 4))
        lines = [ax.plot([], [], label=f'Head {i+1}')[0] for i in range(self.H)]
        ax.set_xlim(0, Q - 1); ax.set_ylim(0, ymax + 0.05)
        ax.set_xticks(np.arange(Q)); ax.set_xticklabels(tokens, rotation=90)
        ax.set_xlabel("Key Position"); ax.set_ylabel("Attention Weight")
        title = ax.set_title(f'Layer {layer_idx+1} — Query[0]')
        ax.legend(fontsize=8, loc='upper right', ncol=2)
        self._apply_gif_layout(fig, top=0.90)

        def init():
            for ln in lines: ln.set_data([], [])
            return lines

        def update(q):
            x = np.arange(Q)
            for h, ln in enumerate(lines):
                ln.set_data(x, frames[q, h])
            title.set_text(f'Layer {layer_idx+1} — Query[{q}]: {tokens[q]}\n"{self.sentences[sent_idx]}"')
            return lines + [title]

        ani = animation.FuncAnimation(fig, update, frames=Q, init_func=init, interval=interval, blit=False)
        if PIL_OK: ani.save(filename, writer='pillow', fps=fps); print(f"Saved: {filename}")
        else: display(HTML(ani.to_jshtml()))
        plt.close(fig)

    # --------------------- Cross-sentence Animation ---------------------
    def _all_sentence_indices(self, sent_indices):
        if self.sentences is None: raise ValueError("Please run analyze_sentences() first.")
        if sent_indices is None: return list(range(len(self.sentences)))
        return list(sent_indices)

    def animate_layer_head_heatmap_across_sentences(self, layer_idx: int, head_idx: int, force: bool = False, sent_indices: Optional[List[int]] = None, interval: int = 900, fps: int = 2):
        sub_dir = self._ensure_dir("appendix/cross_sentence")
        filename = os.path.join(sub_dir, f"XS_L{layer_idx+1}_H{head_idx+1}_across_sentences.gif")
        if os.path.exists(filename) and not force:
            print(f"Skipping, already exists: {filename}"); return
        s_list = self._all_sentence_indices(sent_indices)
        frames, vmin, vmax = [], 1e9, -1e9
        for s_idx in s_list:
            tokens = self.tokens_list[s_idx]
            Q = len(tokens)
            mat = self.attentions_list[s_idx][layer_idx, head_idx]
            frames.append((s_idx, mat, tokens, Q))
            vmin = min(vmin, float(mat.min())); vmax = max(vmax, float(mat.max()))
        if vmax <= vmin: vmax = vmin + 1e-6

        fig, ax = plt.subplots(figsize=(8, 6))

        def update(i):
            ax.clear()
            s_idx, mat, tokens, Q = frames[i]
            im = ax.imshow(mat, vmin=vmin, vmax=vmax, cmap='viridis', aspect='auto')
            fig.colorbar(im, ax=ax, label="Attention Weight")
            ax.set_xticks(np.arange(Q)); ax.set_xticklabels(tokens, rotation=90, fontsize=8)
            ax.set_yticks(np.arange(Q)); ax.set_yticklabels(tokens, fontsize=8)
            ax.set_xlabel("Key"); ax.set_ylabel("Query")
            ax.set_title(f"Sentence {s_idx+1} — Layer {layer_idx+1} Head {head_idx+1}\n\"{self.sentences[s_idx]}\"")
            self._apply_gif_layout(fig, top=0.90)
            return [im]

        ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=interval, blit=False)
        if PIL_OK: ani.save(filename, writer='pillow', fps=fps); plt.close(fig); print(f"Saved: {filename}")
        else: display(HTML(ani.to_jshtml()))
        plt.close(fig)

    def animate_query_entropy_heatmap_across_sentences(self, query_idx: int, force: bool = False, sent_indices: Optional[List[int]] = None, interval: int = 900, fps: int = 2):
        sub_dir = self._ensure_dir("appendix/cross_sentence")
        filename = os.path.join(sub_dir, f"XS_queryEntropy_Q{query_idx}_across_sentences.gif")
        if os.path.exists(filename) and not force:
            print(f"Skipping, already exists: {filename}"); return
        s_list = self._all_sentence_indices(sent_indices)
        frames, gmin, gmax = [], 1e9, -1e9
        for s_idx in s_list:
            tokens = self.tokens_list[s_idx]
            Q = len(tokens)
            q = min(query_idx, Q-1)
            ent = np.zeros((self.L, self.H), dtype=np.float32)
            for l in range(self.L):
                for h in range(self.H):
                    ent[l, h] = sp_entropy(self.attentions_list[s_idx][l, h, q, :], base=2)
            frames.append((s_idx, ent, tokens, q))
            gmin = min(gmin, float(ent.min())); gmax = max(gmax, float(ent.max()))
        if gmax <= gmin: gmax = gmin + 1e-6

        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(frames[0][1], vmin=gmin, vmax=gmax, cmap='magma', aspect='auto')
        plt.colorbar(im, ax=ax).set_label(f"Entropy ({self.base_unit})")
        ax.set_xlabel("Heads"); ax.set_ylabel("Layers")
        ax.set_xticks(np.arange(self.H)); ax.set_xticklabels([f"H{i+1}" for i in range(self.H)])
        ax.set_yticks(np.arange(self.L)); ax.set_yticklabels([f"L{i+1}" for i in range(self.L)])
        title = ax.set_title("")
        self._apply_gif_layout(fig, top=0.90)

        def update(i):
            s_idx, ent, tokens, q = frames[i]
            im.set_data(ent)
            title.set_text(f'Sentence {s_idx+1} — Query[{q}]: {tokens[q]}\n"{self.sentences[s_idx]}"')
            return [im, title]

        ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=interval, blit=False)
        if PIL_OK: ani.save(filename, writer='pillow', fps=fps); plt.close(fig); print(f"Saved: {filename}")
        else: display(HTML(ani.to_jshtml()))
        plt.close(fig)

    def animate_key_mass_heatmap_across_sentences(self, key_idx: int, force: bool = False, sent_indices: Optional[List[int]] = None, interval: int = 900, fps: int = 2):
        sub_dir = self._ensure_dir("appendix/cross_sentence")
        filename = os.path.join(sub_dir, f"XS_keyMass_K{key_idx}_across_sentences.gif")
        if os.path.exists(filename) and not force:
            print(f"Skipping, already exists: {filename}"); return
        s_list = self._all_sentence_indices(sent_indices)
        frames, gmin, gmax = [], 1e9, -1e9
        for s_idx in s_list:
            tokens = self.tokens_list[s_idx]
            Q = len(tokens)
            k = min(key_idx, Q-1)
            M = self.attentions_list[s_idx][:, :, :, k].mean(axis=2)
            frames.append((s_idx, M, tokens, k))
            gmin = min(gmin, float(M.min())); gmax = max(gmax, float(M.max()))
        if gmax <= gmin: gmax = gmin + 1e-6

        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(frames[0][1], vmin=gmin, vmax=gmax, cmap='plasma', aspect='auto')
        plt.colorbar(im, ax=ax).set_label("Avg Attention Received")
        ax.set_xlabel("Heads"); ax.set_ylabel("Layers")
        ax.set_xticks(np.arange(self.H)); ax.set_xticklabels([f"H{i+1}" for i in range(self.H)])
        ax.set_yticks(np.arange(self.L)); ax.set_yticklabels([f"L{i+1}" for i in range(self.L)])
        title = ax.set_title("")
        self._apply_gif_layout(fig, top=0.90)

        def update(i):
            s_idx, M, tokens, k = frames[i]
            im.set_data(M)
            title.set_text(f'Sentence {s_idx+1} — Key[{k}]: "{tokens[k]}" (Avg received)\n"{self.sentences[s_idx]}"')
            return [im, title]

        ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=interval, blit=False)
        if PIL_OK: ani.save(filename, writer='pillow', fps=fps); plt.close(fig); print(f"Saved: {filename}")
        else: display(HTML(ani.to_jshtml()))
        plt.close(fig)

    def animate_multihead_overlay_across_sentences(self, layer_idx: int, query_idx: int, force: bool = False, sent_indices: Optional[List[int]] = None, interval: int = 1100, fps: int = 2):
        sub_dir = self._ensure_dir("appendix/cross_sentence")
        filename = os.path.join(sub_dir, f"XS_L{layer_idx+1}_Q{query_idx}_multihead_overlay_across_sentences.gif")
        if os.path.exists(filename) and not force:
            print(f"Skipping, already exists: {filename}"); return
        s_list = self._all_sentence_indices(sent_indices)
        frames, global_ymax = [], 1e-9
        for s_idx in s_list:
            tokens = self.tokens_list[s_idx]
            Q = len(tokens)
            q = min(query_idx, Q-1)
            curves = self.attentions_list[s_idx][layer_idx, :, q, :]
            frames.append((s_idx, curves, tokens, Q, q))
            global_ymax = max(global_ymax, float(curves.max()))
        if global_ymax <= 0: global_ymax = 1e-6

        fig, ax = plt.subplots(figsize=(10, 4))
        lines = [ax.plot([], [], label=f'Head {i+1}')[0] for i in range(self.H)]
        title = ax.set_title("")
        ax.set_xlabel("Key Position"); ax.set_ylabel("Attention Weight"); ax.set_ylim(0, global_ymax * 1.05)
        ax.legend(fontsize=8, loc='upper right', ncol=2)
        self._apply_gif_layout(fig, top=0.90)

        def update(i):
            s_idx, curves, tokens, Q, q = frames[i]
            x = np.arange(Q)
            ax.set_xlim(-0.5, Q - 0.5)
            ax.set_xticks(x); ax.set_xticklabels(tokens, rotation=90, fontsize=8)
            for h, ln in enumerate(lines):
                y = curves[h] if h < curves.shape[0] else np.zeros(Q)
                ln.set_data(x, y)
            title.set_text(f'Sentence {s_idx+1} — Layer {layer_idx+1} Query[{q}]: {tokens[q]}\n"{self.sentences[s_idx]}"')
            return lines + [title]

        ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=interval, blit=False)
        if PIL_OK: ani.save(filename, writer='pillow', fps=fps); plt.close(fig); print(f"Saved: {filename}")
        else: display(HTML(ani.to_jshtml()))
        plt.close(fig)



In [ ]:
# @title


SAVE_DIRECTORY = "/content/drive/MyDrive/w4_report_output_final"

SENTENCES_TO_ANALYZE = [
    "The quick brown fox chases after the lazy dog.",  # 0: active_passive
    "The lazy dog is chased by the quick brown fox.",  # 1: active_passive
    "What time does the train leave tomorrow morning?",# 2: question
    "He is reading a book in the library.",            # 3: statement
    "Do you know where my keys are?",                  # 4: question
    "The man who lives in the house at the end of the street is a doctor.", # 5: long_dependency
    "The bank is on the river bank.",                  # 6: polysemy
    "He booked a flight to London.",                   # 7: statement
    "He read a book about London.",                    # 8: statement
    "The dog wagged its tail because it was happy.",   # 9: coreference
    "John is a great guy and I really trust him.",     # 10: coreference
    "The movie was not good, it was fantastic.",       # 11: negation
    "What a fantastic movie it was!",                  # 12: statement
    "Was the movie good or fantastic?",                # 13: question
    "I never said she stole my money.",                # 14: negation
    "Paris is the capital of France."                  # 15: fact
]

ADDITIONAL_SENTENCES = [
    "The archer hit the target with an arrow.",
    "The archer missed the target with an arrow."
]

POLYSEMY_PAIRS = [
    # bark
    "The bark of the dog was loud.",
    "The bark of the tree was rough.",
    # bat
    "He swung the bat and hit a home run.",
    "A bat flew out of the cave at dusk.",
    # jam
    "We were stuck in a traffic jam on the way home.",
    "She spread strawberry jam on her toast.",
    # coach
    "The coach praised the team after the match.",
    "We traveled to Oxford by coach.",
    # match
    "The match ended in a draw.",
    "He struck a match to light the candle.",
    # spring
    "Spring is my favorite season.",
    "The mattress has a broken spring.",
    # light
    "The bag is light enough to carry.",
    "Please turn on the light in the hallway.",
    # seal
    "The seal clapped its flippers at the zoo.",
    "Please seal the envelope before you send it."
]

POLYSEMY_MIX = [
    "The bark of the dog, near the tree’s bark, was loud.",
    "She spread strawberry jam while being stuck in a traffic jam.",
    "The coach addressed the team on the coach after the match."
]

EXTRA_POLYSEMY_SENTENCES = [
    # bark: dog vs tree
    "The dog began to bark when the mail carrier arrived.",
    "At night the dogs bark loudly outside the window.",
    "The tree's bark was dark and deeply cracked.",
    "Peel off the loose bark before painting the trunk.",

    # bat: sports vs animal
    "The batter gripped the bat tightly before the pitch.",
    "He bought a new bat for the tournament.",
    "A bat hovered near the porch light.",
    "We saw a colony of bats sleeping in the cave.",

    # jam: traffic vs food
    "There was a severe traffic jam near the bridge.",
    "Morning commuters created a long jam on the highway.",
    "He prefers raspberry jam on warm scones.",
    "The bakery sells homemade apricot jam.",

    # coach: person vs vehicle
    "The coach drew a new play on the board.",
    "Our coach encouraged us to keep practicing.",
    "They boarded a coach to the airport at dawn.",
    "The intercity coach was delayed by heavy rain.",

    # match: sport vs fire
    "The football match kicked off at noon.",
    "Tickets for the final match sold out quickly.",
    "He lit the stove with a wooden match.",
    "Keep matches away from children.",

    # spring: season vs coil
    "Flowers bloom early in spring.",
    "In spring the days get longer.",
    "The door relies on a metal spring to close softly.",
    "A tiny spring popped out of the toy car.",

    # light: weight vs illumination
    "This laptop is light enough for travel.",
    "Use a light backpack to reduce strain.",
    "Switch off the light before leaving.",
    "A soft light filled the studio.",

    # seal: animal vs verb
    "A young seal rested on the beach.",
    "We watched seals diving near the pier.",
    "Seal the bottle tightly to prevent leaks.",
    "Please seal and label each sample bag.",

    # bank: finance vs river
    "The bank closes early on Fridays.",
    "She opened a new account at the bank downtown.",
    "Children played on the sandy river bank.",
    "Wildflowers grew along the river bank."
]
EXTRA_POLYSEMY_LABELS = ["polysemy"] * len(EXTRA_POLYSEMY_SENTENCES)

CROSS_PHENOMENON_SENTENCES = [
    ("Is the bank open today?", "question"),
    ("Did you travel by coach or by train?", "question"),
    ("It is not light in the hallway yet.", "negation"),
    ("Why was there a traffic jam this morning?", "question"),
    ("Did the dog bark all night?", "question"),
    ("He did not find a match to light the candle.", "negation"),
    ("When does spring begin this year?", "question"),
    ("Did you seal the envelope before mailing it?", "question"),
    ("Was that a bat flying over the park?", "question")
]


FINAL_SENTENCES = (
    SENTENCES_TO_ANALYZE
    + ADDITIONAL_SENTENCES
    + POLYSEMY_PAIRS
    + POLYSEMY_MIX
    + EXTRA_POLYSEMY_SENTENCES
    + [s for s, _ in CROSS_PHENOMENON_SENTENCES]
)


SENTENCE_GROUPS = [
    "active_passive", "active_passive", "question", "statement", "question",
    "long_dependency", "polysemy", "statement", "statement", "coreference",
    "coreference", "negation", "statement", "question", "negation", "fact",
    "minimal_pair", "minimal_pair"
]
SENTENCE_GROUPS += ["polysemy"] * len(POLYSEMY_PAIRS)
SENTENCE_GROUPS += ["ignore"] * len(POLYSEMY_MIX)
SENTENCE_GROUPS += EXTRA_POLYSEMY_LABELS
SENTENCE_GROUPS += [g for _, g in CROSS_PHENOMENON_SENTENCES]

assert len(FINAL_SENTENCES) == len(SENTENCE_GROUPS), \
    f"Mismatched number of labels: {len(FINAL_SENTENCES)} sentences vs {len(SENTENCE_GROUPS)} groups"

report_visualizer = ReportReadyVisualizer(save_dir=SAVE_DIRECTORY)
report_visualizer.analyze_sentences(sentences=FINAL_SENTENCES)

H_lh_mean = report_visualizer.plot_figure_1_global_entropy_heatmap(force=False, show=False)

print(f"There are {len(FINAL_SENTENCES)} sentences and {len(SENTENCE_GROUPS)} labels.")


In [ ]:
# @title
# ===================================================================
print("--- Token/Index Inspector ---")
if 'report_visualizer' in locals() and hasattr(report_visualizer, 'sentences') and report_visualizer.sentences is not None:
  for s_idx, s in enumerate(report_visualizer.sentences):
    tokens = report_visualizer.tokens_list[s_idx]
    print(f"\n--- Sentence {s_idx}: {s} ---")
    print(list(enumerate(tokens)))
else:
  print("Error: Please ensure that you have successfully run the cell for defining and analyzing sentences.")



In [ ]:

print("\n" + "="*80)
print(">>> Generating core charts...")
print("="*80)

H_lh_mean = report_visualizer.plot_figure_1_global_entropy_heatmap(force=True, show=True)


In [ ]:
report_visualizer.plot_figure_2_layerwise_entropy_stats_ci(
    H_lh_mean, force=True, show=True, conf_level=0.95, ci_basis="heads"
)


In [ ]:

report_visualizer.plot_figure_2_layerwise_entropy_stats_ci(
    H_lh_mean, force=True, show=True, conf_level=0.95, ci_basis="sentences"
)

In [ ]:
# @title
def _idx(s_idx, tok):
    return report_visualizer.find_token_index(report_visualizer.tokens_list[s_idx], tok, prefer_exact=True)

# --- Case 1---
print("--- Active ---")
report_visualizer.animate_in_main_dir(
    sent_idx=0,
    anim_type="key_from_queries",
    phenomenon_name="Active_Voice_chases_evolution",
    key_idx=_idx(0, "chases"),
    mode="per-layer",
    force=True
)

print("\n--- Passive ---")
report_visualizer.animate_in_main_dir(
    sent_idx=1,
    anim_type="key_from_queries",
    phenomenon_name="Passive_Voice_chased_evolution",
    key_idx=_idx(1, "chased"),
    mode="per-layer",
    force=True
)


# ---Case 2 ---
print("\n--- its ---")
report_visualizer.animate_in_main_dir(
    sent_idx=9,
    anim_type="query_to_others",
    phenomenon_name="Coreference_its_to_dog",
    query_idx=_idx(9, "its"),
    mode="per-layer",
    force=True
)
print("\n--- it ---")
report_visualizer.animate_in_main_dir(
    sent_idx=9,
    anim_type="query_to_others",
    phenomenon_name="Coreference_it_to_dog",
    query_idx=_idx(9, "it"),
    mode="per-layer",
    force=True
)


# ---Case 3 ---
print("\n--- bank 1 ---")
s_bank = 6
bank1 = _idx(s_bank, "bank")
toks = report_visualizer.tokens_list[s_bank]
try:
    bank2 = toks.index("bank", bank1 + 1)
except ValueError:
    bank2 = 7

report_visualizer.animate_in_main_dir(
    sent_idx=s_bank,
    anim_type="key_from_queries",
    phenomenon_name="Polysemy_queries_to_bank1",
    key_idx=bank1,
    mode="LH-mean",
    force=True
)

print("\n--- bank 2 ---")
report_visualizer.animate_in_main_dir(
    sent_idx=s_bank,
    anim_type="key_from_queries",
    phenomenon_name="Polysemy_queries_to_bank2",
    key_idx=bank2,
    mode="LH-mean",
    force=True
)

In [ ]:
# @title
def _pick_head_idx_for_layer(H_lh_mean, layer_idx, choice='high'):
    low, mid, high = report_visualizer.pick_heads_by_entropy(H_lh_mean[layer_idx])
    return {'low': low, 'mid': mid, 'high': high}[choice]

def jsd_pairwise_matrix(vecs, base=2):
    S = len(vecs); D = np.zeros((S, S), dtype=np.float64)
    for i in range(S):
        for j in range(i, S):
            d = jensenshannon(vecs[i], vecs[j], base=base)**2
            D[i, j] = d; D[j, i] = d
    return D

def summarize_jsd_by_group(D, groups, subset_indices):
    labs = [groups[i] for i in subset_indices]
    keep = [i for i,g in enumerate(labs) if g != "ignore"]
    if len(keep) < 2: return np.array([]), np.array([]), []
    intra, inter = [], []
    for a in range(len(keep)):
        for b in range(a+1, len(keep)):
            i, j = keep[a], keep[b]
            if labs[keep[a]] == labs[keep[b]]: intra.append(D[i, j])
            else:                              inter.append(D[i, j])
    return np.array(intra), np.array(inter), keep

def _pick_head_idx_for_layer(H_lh_mean, layer_idx, choice='high'):
    low, mid, high = report_visualizer.pick_heads_by_entropy(H_lh_mean[layer_idx])
    return {'low': low, 'mid': mid, 'high': high}[choice]

def _plot_delta_curve(delta, title):
    x = np.arange(1, len(delta)+1)
    plt.figure(figsize=(7,3.6))
    plt.plot(x, delta, marker='o'); plt.axhline(0, linestyle='--', alpha=0.5)
    plt.xlabel("Layer"); plt.ylabel("ΔJSD (inter − within)")
    plt.title(title); plt.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout(); plt.show()

def _pick_top_layers(delta, k=3, min_gap=1):
    idx = np.argsort(np.nan_to_num(delta, nan=-1))[::-1]
    chosen = []
    for l in idx:
        if np.isnan(delta[l]) or delta[l] <= 0: continue
        if all(abs(l - c) >= min_gap for c in chosen):
            chosen.append(l)
        if len(chosen) >= k: break
    return chosen

def plot_pairwise_heatmap(D, title, subset_indices):
    fig, ax = plt.subplots(figsize=(6.2, 5.0))
    sns.heatmap(D, vmin=0, vmax=1, cmap="viridis", square=True, cbar_kws={'label': 'JSD (bits, base-2)'})
    ax.set_title(title, pad=10)
    ax.set_xlabel("Sentence idx (subset)"); ax.set_ylabel("Sentence idx (subset)")
    tick = np.arange(len(subset_indices))
    ax.set_xticks(tick); ax.set_yticks(tick)
    ax.set_xticklabels(subset_indices, rotation=0); ax.set_yticklabels(subset_indices, rotation=0)
    plt.tight_layout(); plt.show()

def plot_jsd_distributions(intra, inter, title):
    if len(intra)==0 and len(inter)==0:
        print("[WARN] No valid group comparisons were found：skipped."); return
    fig, ax = plt.subplots(figsize=(5.4, 4.0))
    data, names = [], []
    if len(intra): data.append(intra); names.append('Within-phenomenon')
    if len(inter): data.append(inter); names.append('Across-phenomena')
    ax.violinplot(data, showmeans=True, showextrema=True)
    ax.set_xticks(np.arange(1, len(data)+1)); ax.set_xticklabels(names)
    ax.set_ylabel('JSD (bits, base-2)')
    ax.set_title(title, pad=10); ax.grid(True, axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout(); plt.show()

# ---------- A) Collect tokenset hits: Supports match_mode='lenient' ----------
def _collect_dists_for_tokenset(token_set, layer_idx: int, head_choice='mean',
                                agg='first', pad_mode='right', match_mode='strict'):
    token_set = set(token_set)

    def _hits(toks):
        if match_mode == 'strict':
            return [i for i,t in enumerate(toks) if t in token_set]
        # lenient: Expands not/never/n't and performs case-insensitive matching.
        expand = set()
        for tok in token_set:
            tl = tok.lower()
            if tl in {"not","never","n't"}:
                expand.update({"not","never","n't"})
            else:
                expand.add(tl)
        return [i for i,t in enumerate(toks) if t.lower() in expand]

    subset, vecs, maxK = [], [], 0
    for s_idx, toks in enumerate(report_visualizer.tokens_list):
        q_idxs = _hits(toks)
        if not q_idxs:
            continue
        subset.append(s_idx)
        A = report_visualizer.attentions_list[s_idx][layer_idx]   # [H,Q,K]
        if   agg == 'first': qs = [q_idxs[0]]
        elif agg == 'last' : qs = [q_idxs[-1]]
        elif agg == 'mean' : qs = q_idxs
        else: raise ValueError("agg must be 'first'|'last'|'mean'")
        if head_choice == 'mean':
            v = A[:, qs, :].mean(axis=0).mean(axis=0).astype(np.float64)
        else:
            h = _pick_head_idx_for_layer(H_lh_mean, layer_idx, head_choice)
            v = A[h, qs, :].mean(axis=0).astype(np.float64)
        v = np.clip(v, 0, None); s = v.sum()
        v = (np.ones_like(v)/len(v)) if s<=0 else (v/s)
        vecs.append(v); maxK = max(maxK, v.shape[0])

    if len(subset) < 2:
        return None, None

    padded = []
    for v in vecs:
        if v.shape[0] < maxK:
            pad = maxK - v.shape[0]
            v = np.pad(v, (0, pad), constant_values=0.0) if pad_mode=='right' else np.pad(v, (pad, 0), constant_values=0.0)
        padded.append(v)
    return padded, subset

# ---------- B) ΔJSD: When inter is empty, use bootstrap as a fallback. ----------
def _delta_jsd_over_layers(collector_fn, *args, fallback_inter='none', **kwargs):
    L = report_visualizer.L
    delta = np.full(L, np.nan); intra_med = np.full(L, np.nan); inter_med = np.full(L, np.nan); valid = np.zeros(L, bool)
    rng = np.random.default_rng(20240921)
    for l in range(L):
        out = collector_fn(*args, layer_idx=l, **kwargs)
        if out is None or out[0] is None:
            continue
        padded, subset = out
        D = jsd_pairwise_matrix(padded, base=2)
        intra, inter, keep = summarize_jsd_by_group(D, SENTENCE_GROUPS, subset)

        # When inter is empty, perform random binary search; bootstrap approximates cross-group search.
        if (len(intra)==0 or len(inter)==0) and fallback_inter == 'bootstrap':
            m = len(subset)
            if m >= 4:
                idx = np.arange(m); rng.shuffle(idx)
                g1 = idx[:m//2]; g2 = idx[m//2:]
                inter = np.array([D[i,j] for i in g1 for j in g2], dtype=np.float64)
                intra = np.array(
                    [D[i,j] for a in (g1, g2) for i in a for j in a if j>i],
                    dtype=np.float64
                )
            else:
                continue

        if len(intra)==0 or len(inter)==0:
            continue
        im, em = np.median(intra), np.median(inter)
        intra_med[l], inter_med[l], delta[l], valid[l] = im, em, (em-im), True
    return delta, intra_med, inter_med, valid

def run_jsd_tokenset_multi(token_set, label, head_choices=('mean','low','mid','high'),
                           topk=3, min_gap=1, agg='first', match_mode='lenient',
                           fallback_inter='bootstrap'):
    print(f"\n=== [JSD] phenomenon = {label}, token_set = {token_set} ===")
    delta, *_ = _delta_jsd_over_layers(
        _collect_dists_for_tokenset, set(token_set),
        head_choice='mean', agg=agg, match_mode=match_mode,
        fallback_inter=fallback_inter
    )
    _plot_delta_curve(delta, f'ΔJSD across layers — {label}')
    layers = _pick_top_layers(delta, k=topk, min_gap=min_gap)
    if not layers:
        print("[WARN] No valid layers available (ΔJSD<=0 or insufficient samples)");
        return
    print("Auto-picked layers (1-based):", [l+1 for l in layers])

    for head_choice in head_choices:
        for l in layers:
            out = _collect_dists_for_tokenset(set(token_set), l,
                                              head_choice=head_choice, agg=agg,
                                              match_mode=match_mode)
            if out is None or out[0] is None:
                print(f"[WARN] {label} @ L{l+1} — insufficient samples");
                continue
            padded, subset = out
            D = jsd_pairwise_matrix(padded, base=2)
            plot_pairwise_heatmap(D, title=f'Pairwise JSD — {label} @ L{l+1}-{head_choice}', subset_indices=subset)
            intra, inter, _ = summarize_jsd_by_group(D, SENTENCE_GROUPS, subset)
            plot_jsd_distributions(intra, inter, title=f'JSD — {label} @ L{l+1}-{head_choice}')


In [ ]:
def run_jsd_token_multi(token_str, head_choices=('mean','low','mid','high'),
                        topk=3, min_gap=1, agg='first',
                        match_mode='strict', fallback_inter='bootstrap'):
    run_jsd_tokenset_multi(
        token_set={token_str},
        label=token_str,
        head_choices=head_choices,
        topk=topk,
        min_gap=min_gap,
        agg=agg,
        match_mode=match_mode,
        fallback_inter=fallback_inter
    )

In [ ]:
TOKENS_FOR_JSD = ["bank","bark","bat","jam","coach","match","spring","light","seal","dog"]
for tok in TOKENS_FOR_JSD:
    run_jsd_token_multi(tok, head_choices=('mean','low','mid','high'),
                        topk=3, min_gap=1, agg='first')


In [ ]:

# --- II. Appendix/Exploratory Analysis ---

# # -- 2.1 Appendix Global Graph --

# print("\n--- Generate the global graph in the appendix ---")
report_visualizer.plot_global_mass_to_token()
report_visualizer.plot_global_specialization()

# # -- 2.2 Batch Static Images in Appendix --
# print("\n--- Generate batch static images in the appendix (may take a long time) ---")
report_visualizer.plot_all_layer_grids()
report_visualizer.plot_all_head_similarities()


In [ ]:
# =========================

# Generate cross-sentence animation (negation subset + L1/L6/L12 × low/medium/high head)

# =========================

rv = report_visualizer

neg_indices = []
for s_idx, toks in enumerate(rv.tokens_list):
    if any(t.lower() in {"not","never","n't"} for t in toks):
        neg_indices.append(s_idx)

if len(neg_indices) < 2:
    print("[WARN] The number of negation subsets is insufficient, less than 2; cross-sentence animation may be meaningless.")

layers_for_demo = [0, rv.L//2, rv.L-1]  # [0,6,11]

if 'H_lh_mean' not in globals():
    H_lh_mean = rv.plot_figure_1_global_entropy_heatmap(force=False, show=False)

def _heads_for_layer(layer):
    low, mid, high = rv.pick_heads_by_entropy(H_lh_mean[layer])
    return [('low', low), ('mid', mid), ('high', high)]

for L in layers_for_demo:
    for tag, H in _heads_for_layer(L):
        print(f"Making XS heatmap: L{L+1} H{H+1} ({tag}) on negation subset {neg_indices}")
        rv.animate_layer_head_heatmap_across_sentences(
            layer_idx=L, head_idx=H, sent_indices=neg_indices, force=False, interval=900, fps=2
        )

In [ ]:

# # -- 2.3 Appendix Animation (Select a sentence and layer/head as an example) --

# print("\n--- Generate the animation example in the appendix ---")

# report_visualizer.animate_heads_in_layer(sent_idx=0, layer_idx=11)

# report_visualizer.animate_layers_for_head(sent_idx=0, head_idx=5)

# report_visualizer.animate_query_to_others(sent_idx=5, query_idx=15, mode="per-layer") # Animate demonstrating long-distance dependencies

# report_visualizer.animate_key_from_queries(sent_idx=6, key_idx=6, mode="LH-mean") # Static graph showing who follows the second bank

# # -- 2.4 Appendix Cross-Sentence Animation (Select a comparison as an example) --

# print("\n--- Generate the cross-sentence animation example in the appendix) ---")